# RecFlowTime — conditional generation

Imputation and forecasting from a model trained unconditionally, with **no
retraining and no second model**.

Because the forward process is the straight interpolation
$x_t = (1-t)x + t\varepsilon$, the observed entries of a target can be carried to
the noise level of the current integration step in one line and written back
after every Euler step. At $t=0$ they are reproduced exactly. Nothing about the
model changes and no gradient is taken, so sampling still costs the same 20
function evaluations as unconditional generation.

Run `01_train_and_sample.ipynb` first, or point `CKPT` at any checkpoint it
wrote. This notebook reproduces the conditional-generation appendix of the
paper.


## 1. Setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("."))
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import torch
import matplotlib.pyplot as plt

from recflowtime.config import DenoiserConfig, CoreConfig
from recflowtime.denoiser import TransformerDenoiser
from recflowtime.flow import RectifiedFlow
from recflowtime.conditional import (impute_mask, forecast_mask, complete_k,
                                     linear_fill, last_value_fill, scores)

# ---------------------------------------------------------------- choose here
DATASET = "sines"          # "sines" | "stocks" | "energy" | "ecg"
CKPT    = f"results/notebook/{DATASET}_ours_ckpt.pt"  # written by notebook 01
STEPS   = 20               # Euler steps: the same budget as unconditional
K       = 10               # completions drawn per masked input
N       = 200              # held-out sequences to score
SEED    = 0

## 2. Load the trained model

Sampling uses the EMA weights, as in the paper.

In [ ]:
blob = torch.load(CKPT, map_location="cpu", weights_only=False)
dcfg  = DenoiserConfig(**blob["config"]["denoiser"])
ccfg  = CoreConfig(**blob["config"]["core"])

model = TransformerDenoiser(dcfg)
model.load_state_dict(blob["ema_state"])
model.eval()
core = RectifiedFlow(ccfg, device="cpu")

real = np.load(f"data/splits/{DATASET}_T{dcfg.seq_len}.npz")["test"][:N]
print(f"{DATASET}: T={dcfg.seq_len}, F={dcfg.n_features}, {len(real)} test sequences")

## 3. Imputation

Entries are hidden in geometric streaks of mean length three, at the same
timesteps across all features. Contiguous streaks are used rather than isolated
points because an isolated hidden point is recoverable by interpolation from its
neighbours and would flatter any method.

Three numbers are reported, and they answer different questions. `mse` scores a
single draw, which penalises a calibrated generative model by construction,
since squared error is minimised by the conditional *mean* and a sample is not
it. `mse_mean` scores the K-sample mean, which estimates that conditional mean
and is what linear interpolation already computes. `crps` is a proper scoring
rule. Linear interpolation is shown as a scale, not as a competing method: on a
near-random walk it *is* the posterior mean.

In [ ]:
rng  = np.random.default_rng(SEED)
rows = []
for ratio in (0.1, 0.3, 0.5, 0.7, 0.9):
    keep  = impute_mask(real.shape, ratio, "concurrent", 3, np.random.default_rng(SEED))
    preds = complete_k(model, core, dcfg, real * keep, keep, STEPS, K, seed=SEED)
    s = scores(preds, real, keep)
    s["ratio"] = ratio
    s["linear"] = float((((linear_fill(real * keep, keep) - real)[~keep]) ** 2).mean())
    rows.append(s)
    # the streak process hits `ratio` only approximately, so print what was
    # actually hidden rather than what was asked for
    print(f"asked {ratio:.0%} / hidden {(~keep).mean():.0%}   mse(1 draw) {s['mse']:.4f}"
          f"   mse({K}-mean) {s['mse_mean']:.4f}   crps {s['crps']:.4f}"
          f"   linear {s['linear']:.4f}   obs err {s['obs_max_abs']:.0e}")

The last column is a check rather than a result: observed entries are
written back exactly, so it must be zero.

## 4. Forecasting

The same procedure with a different mask — the final $h$ steps of every feature
are hidden. Repeating the last observed value is the reference here, and it is
the optimal forecast for a random walk.

In [ ]:
T = dcfg.seq_len
for frac in (0.125, 0.25, 0.5):
    h     = max(1, int(round(frac * T)))
    keep  = forecast_mask(real.shape, h)
    preds = complete_k(model, core, dcfg, real * keep, keep, STEPS, K, seed=SEED)
    s = scores(preds, real, keep)
    ref = float((((last_value_fill(real, h) - real)[~keep]) ** 2).mean())
    print(f"h = T/{int(round(1/frac))} ({h:3d} steps)  mse({K}-mean) {s['mse_mean']:.4f}"
          f"   crps {s['crps']:.4f}   last-value {ref:.4f}")

## 5. Predictive spread

Every completion of one input, drawn as a band. The width is the point: a narrow
band means the model is confident, and a wide one means the conditional
distribution really is broad — which is the correct behaviour on data like
Stocks, and the reason a proper scoring rule is needed to see it.

In [ ]:
def show(keep, title, n_show=2, feat=0):
    preds = complete_k(model, core, dcfg, real * keep, keep, STEPS, K, seed=SEED)
    med   = np.median(preds[:, :, :, feat], axis=0)
    hid   = ~keep[:, :, feat]
    err   = ((med - real[:, :, feat]) ** 2 * hid).sum(1) / np.maximum(hid.sum(1), 1)
    rngf  = real[:, :, feat].max(1) - real[:, :, feat].min(1)
    act   = np.where(rngf >= np.median(rngf))[0]          # skip near-flat windows
    picks = act[np.argsort(err[act])[:n_show]]

    fig, axes = plt.subplots(1, n_show, figsize=(5.2 * n_show, 2.2), squeeze=False)
    for ax, i in zip(axes[0], picks):
        P = preds[:, i, :, feat]
        ax.fill_between(np.arange(T), np.percentile(P, 10, 0), np.percentile(P, 90, 0),
                        color="#1C7ED6", alpha=.28, lw=0, label="10th-90th pct")
        ax.plot(real[i, :, feat], color="#E03131", lw=2.0, alpha=.45, label="ground truth")
        ax.plot(np.median(P, 0), color="#1C7ED6", lw=.9, label="median completion")
        for st in np.where(~keep[i, :, feat])[0]:
            ax.axvspan(st - .5, st + .5, color="#F1F3F5", lw=0, zorder=0)
        ax.set_xticks([]); ax.set_yticks([])
    axes[0][0].legend(fontsize=7, frameon=False, loc="lower left")
    fig.suptitle(title); fig.tight_layout(); plt.show()

show(impute_mask(real.shape, 0.5, "concurrent", 3, np.random.default_rng(SEED)),
     "Imputation, 50% hidden")
show(forecast_mask(real.shape, max(1, T // 4)), "Forecasting, h = T/4")

## 6. Does conditioning need more steps?

No. Error falls steeply up to 20 steps and changes little beyond, so the
operating point used everywhere else in the paper is adequate here too.

In [ ]:
keep = impute_mask(real.shape, 0.5, "concurrent", 3, np.random.default_rng(SEED))
for steps in (5, 10, 20, 32):
    preds = complete_k(model, core, dcfg, real * keep, keep, steps, 4, seed=SEED)
    print(f"{steps:3d} steps   mse(4-mean) {scores(preds, real, keep)['mse_mean']:.4f}")

## Notes

**Any mask works.** The mask never enters training — it is applied only inside
the sampling loop — so the model has nothing specialised to a particular
observation pattern. Irregular sampling, a whole feature missing, scattered
gaps, a forecast tail or any combination can be passed at inference.

**What this does not do.** Because nothing is trained for masking, the model has
no mechanism for reconciling generated content with observed content at the
seams. At high hidden fractions the completions can carry more high-frequency
content than the data, which the paper measures as an increase in spectral
distance. Gradient guidance would address this, at the cost of extra function
evaluations per step.
